# 04 - LSTM y GRU Multicapa
## Clasificación de Marcha Patológica

**Autor:** Weimar Andres Arenas Gonzalez  
**Curso:** Fundamentos de Deep Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WeimarArenas/ProyectoDL20261/blob/main/04%20-%20LSTM_GRU.ipynb)

---
**Objetivo:** Implementar y comparar LSTM y GRU de 4 capas, replicando exactamente la arquitectura del paper Jun et al. (2020).

**Referencias del paper:**
- GRU (4 capas, todos joints): **90.13%**
- GRU (4 capas, solo piernas): **93.67%** ← Mejor resultado del paper
- LSTM (4 capas, todos joints): **87.25%**

In [ ]:
import sys

# TensorFlow requiere Python 3.10-3.12
if sys.version_info >= (3, 13):
    raise EnvironmentError(
        "Python " + str(sys.version_info.major) + "." + str(sys.version_info.minor) +
        " no es compatible con TensorFlow.
"
        "Opciones:
"
        "  1. Ejecuta este notebook en Google Colab (recomendado, tiene GPU gratuita)
"
        "  2. Cambia el kernel a Python 3.10, 3.11 o 3.12"
    )

import subprocess
pkgs = ["numpy", "pandas", "matplotlib", "seaborn", "scikit-learn", "tensorflow"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs,
               capture_output=True, timeout=300)
print(f"Python {sys.version_info.major}.{sys.version_info.minor} - Dependencias listas.")


## 1. Configuración del entorno

In [ ]:
import sys, os, re, time, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, confusion_matrix
from collections import Counter


%matplotlib inline
plt.rcParams['figure.figsize'] = (13, 5)
sns.set_style('whitegrid')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

N_CLASSES = 6
CLASS_NAMES = {0: 'Normal', 1: 'Antálgica', 2: 'Rígida', 3: 'Oscilante', 4: 'Steppage', 5: 'Trendelenburg'}
LEG_JOINT_INDICES = [0, 1, 12, 13, 14, 15, 16, 17, 18, 19]

print(f'TensorFlow: {tf.__version__}')
print(f'GPU disponible: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# ============================================================
# FUNCIONES DE CARGA — Formato real del dataset GIST
# CSV: tab-separado, sin header, 101 columnas
# Col 0: timestamp | cols (1+j*4)+1,+2,+3 = x,y,z para joint j
# Directorio padre: human{N}_{gait_type}{rep}/
# ============================================================

import re as _re

# Indices de features: x,y,z de cada articulacion (omite timestamp y joint_id)
FEATURE_COLS = []
for _j in range(25):
    _base = 1 + _j * 4
    FEATURE_COLS.extend([_base + 1, _base + 2, _base + 3])
# Resultado: [2,3,4, 6,7,8, ..., 98,99,100]  -> 75 valores por fila

_GAIT_PAT = _re.compile(
    r'human(\d+)_(normal|antalgic|lurch|steppage|stiff_legged|trendelenburg)\d+'
)
CLASS_MAP = {
    'normal': 0, 'antalgic': 1, 'stiff_legged': 2,
    'lurch': 3, 'steppage': 4, 'trendelenburg': 5,
}
CLASS_NAMES = ['Normal', 'Antalgic', 'Stiff-legged', 'Lurching', 'Steppage', 'Trendelenburg']


def extract_label_and_subject(filepath):
    """Extrae clase y sujeto del nombre del directorio padre del CSV."""
    dirname = os.path.basename(os.path.dirname(filepath))
    m = _GAIT_PAT.match(dirname)
    if not m:
        return None, None
    return CLASS_MAP.get(m.group(2)), int(m.group(1))


def load_csv_sequence(filepath, skip_frames=10, n_frames=50):
    """
    Lee un CSV del GIST dataset (tab-separado, 101 cols, sin header).
    Retorna array (n_frames, 75) omitiendo los primeros skip_frames.
    """
    try:
        df = pd.read_csv(filepath, header=None, sep='\t')
        if df.shape[1] != 101 or len(df) < skip_frames + n_frames:
            return None
        seq = df.iloc[skip_frames:skip_frames + n_frames, FEATURE_COLS].values.astype(np.float32)
        return None if np.any(np.isnan(seq)) else seq
    except Exception:
        return None


def load_full_dataset(data_dir, verbose=True):
    """
    Carga todos los CSV validos del GIST dataset.
    data_dir: ruta a Pathological_Gaits/
    Retorna: X (N,50,75), y (N,), S (N,)
    """
    sequences, labels, subjects = [], [], []
    skipped = 0
    for dirname in sorted(os.listdir(data_dir)):
        dirpath = os.path.join(data_dir, dirname)
        if not os.path.isdir(dirpath):
            continue
        for fname in sorted(os.listdir(dirpath)):
            if not fname.endswith('.csv'):
                continue
            fpath = os.path.join(dirpath, fname)
            label, subject = extract_label_and_subject(fpath)
            if label is None:
                skipped += 1
                continue
            seq = load_csv_sequence(fpath)
            if seq is None:
                skipped += 1
                continue
            sequences.append(seq)
            labels.append(label)
            subjects.append(subject)
    if verbose:
        print(f'Secuencias cargadas: {len(sequences)} | Descartadas: {skipped}')
    return (np.array(sequences, dtype=np.float32),
            np.array(labels, dtype=np.int32),
            np.array(subjects, dtype=np.int32))


In [ ]:
# ---- Cargar datos (desde .npy cache o desde CSV) ----------------------------
if not os.path.exists('pathological_gait_datasets'):
    os.system('git clone --quiet https://github.com/kooksung/pathological_gait_datasets.git')

DATA_DIR = 'pathological_gait_datasets/Pathological_Gaits'

if os.path.exists('X_all.npy') and os.path.exists('y_labels.npy'):
    X    = np.load('X_all.npy')
    y    = np.load('y_labels.npy')
    S    = np.load('S_subjects.npy')
    print(f'Datos cargados desde .npy: X={X.shape}')
else:
    assert os.path.isdir(DATA_DIR), f'No se encontro el dataset en: {DATA_DIR}'
    X, y, S = load_full_dataset(DATA_DIR)
    np.save('X_all.npy', X)
    np.save('y_labels.npy', y)
    np.save('S_subjects.npy', S)
    print(f'Datos cargados desde CSV y guardados: X={X.shape}')

# Seleccion de joints de piernas
LEG_JOINT_INDICES = [0, 1, 12, 13, 14, 15, 16, 17, 18, 19]
leg_feat = [f for j in LEG_JOINT_INDICES for f in [j*3, j*3+1, j*3+2]]
X_legs = X[:, :, leg_feat]

print(f'X={X.shape} | X_legs={X_legs.shape}')
print(f'Clases: {dict(zip(*np.unique(y, return_counts=True)))}')
print(f'Sujetos: {sorted(np.unique(S))}')


In [ ]:
# ── Modo de entrenamiento ────────────────────────────────────────────────────
# QUICK_MODE = True  → rapido en CPU/VSCode  (~2-3 min por fold)
# QUICK_MODE = False → completo en Colab GPU (~1-2 min por fold, mejores resultados)
QUICK_MODE = True

N_EPOCHS   = 20  if QUICK_MODE else 500
PATIENCE   = 5   if QUICK_MODE else 30
BATCH_SIZE = 256 if QUICK_MODE else 64

print(f"Modo: {"RAPIDO (CPU)" if QUICK_MODE else "COMPLETO (GPU/Colab)"}")
print(f"Epocas max={N_EPOCHS} | Patience={PATIENCE} | Batch={BATCH_SIZE}")


## 2. Arquitecturas LSTM y GRU

### Arquitectura del paper (Jun et al. 2020, Fig. 3):
```
Input (50, 75)
  → ReLU Dense (proyección a n_units)
  → GRU/LSTM × 4 capas (125 neuronas c/u)
  → Última capa: solo hidden state final (no return_sequences)
  → Dense(125, ReLU)
  → Softmax(6)
```

In [ ]:
def build_gru_model(input_shape, n_classes=6, n_layers=4, n_units=125,
                    dropout_rate=0.3, l2_reg=1e-4):
    """
    GRU de n_layers capas — Replicación exacta de Jun et al. (2020).
    El clasificador GRU alcanzó 90.13% (todos joints) y 93.67% (solo piernas).
    """
    reg = keras.regularizers.l2(l2_reg)
    
    inputs = layers.Input(shape=input_shape, name='input')
    # Proyección lineal con ReLU — corresponde a 'a_t' en el paper (Eq. 1)
    x = layers.Dense(n_units, activation='relu', kernel_regularizer=reg, name='relu_proj')(inputs)
    
    for i in range(n_layers):
        return_seq = (i < n_layers - 1)
        x = layers.GRU(
            n_units,
            return_sequences=return_seq,
            kernel_regularizer=reg,
            recurrent_regularizer=reg,
            name=f'gru_{i+1}'
        )(x)
        if return_seq:
            x = layers.Dropout(dropout_rate, name=f'drop_{i+1}')(x)
    
    # Fully connected + Softmax (Eq. 5 del paper)
    x = layers.Dense(n_units, activation='relu', kernel_regularizer=reg, name='fc')(x)
    x = layers.Dropout(dropout_rate, name='drop_fc')(x)
    outputs = layers.Dense(n_classes, activation='softmax', name='softmax')(x)
    
    return keras.Model(inputs, outputs, name='GRU_Classifier')


def build_lstm_model(input_shape, n_classes=6, n_layers=4, n_units=125,
                     dropout_rate=0.3, l2_reg=1e-4):
    """
    LSTM de n_layers capas — comparado en el paper: 87.25% (todos joints).
    LSTM tiene más parámetros que GRU (gates forget + input vs. reset + update).
    """
    reg = keras.regularizers.l2(l2_reg)
    
    inputs = layers.Input(shape=input_shape, name='input')
    x = layers.Dense(n_units, activation='relu', kernel_regularizer=reg, name='relu_proj')(inputs)
    
    for i in range(n_layers):
        return_seq = (i < n_layers - 1)
        x = layers.LSTM(
            n_units,
            return_sequences=return_seq,
            kernel_regularizer=reg,
            recurrent_regularizer=reg,
            name=f'lstm_{i+1}'
        )(x)
        if return_seq:
            x = layers.Dropout(dropout_rate, name=f'drop_{i+1}')(x)
    
    x = layers.Dense(n_units, activation='relu', kernel_regularizer=reg, name='fc')(x)
    x = layers.Dropout(dropout_rate, name='drop_fc')(x)
    outputs = layers.Dense(n_classes, activation='softmax', name='softmax')(x)
    
    return keras.Model(inputs, outputs, name='LSTM_Classifier')


# Mostrar resumen de ambas arquitecturas
print('=== Arquitectura GRU (todos joints) ===')
gru_demo = build_gru_model((50, 75))
gru_demo.summary()
print(f'\nParámetros totales GRU: {gru_demo.count_params():,}')

In [ ]:
print('=== Arquitectura LSTM (todos joints) ===')
lstm_demo = build_lstm_model((50, 75))
lstm_demo.summary()
print(f'\nParámetros totales LSTM: {lstm_demo.count_params():,}')

# Comparación de parámetros
print('\n=== Comparación de parámetros ===')
print(f'GRU  (75 feat): {gru_demo.count_params():,}')
print(f'LSTM (75 feat): {lstm_demo.count_params():,}')
print(f'GRU  (30 feat): {build_gru_model((50, 30)).count_params():,}')
print(f'LSTM (30 feat): {build_lstm_model((50, 30)).count_params():,}')

## 3. Funciones de entrenamiento y evaluación

In [ ]:
def evaluate_fold(model, X_test, y_test):
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    n_cls = N_CLASSES
    sensitivity = np.zeros(n_cls)
    specificity = np.zeros(n_cls)
    precision = np.zeros(n_cls)
    for cls in range(n_cls):
        TP = cm[cls, cls]
        FN = cm[cls, :].sum() - TP
        FP = cm[:, cls].sum() - TP
        TN = cm.sum() - TP - FN - FP
        sensitivity[cls] = TP / (TP + FN) if (TP + FN) > 0 else 0
        specificity[cls] = TN / (TN + FP) if (TN + FP) > 0 else 0
        precision[cls]   = TP / (TP + FP) if (TP + FP) > 0 else 0
    return {'accuracy': acc, 'confusion_matrix': cm,
            'sensitivity': sensitivity, 'specificity': specificity, 'precision': precision}


def train_loso_cv(X_data, y, S, model_builder_fn, model_name,
                  n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, patience=PATIENCE):
    logo = LeaveOneGroupOut()
    results = []
    print(f'\n=== LOSO-CV: {model_name} ===')
    
    for fold_idx, (train_idx, test_idx) in enumerate(logo.split(X_data, y, groups=S)):
        test_subj = int(np.unique(S[test_idx])[0])
        N_tr, T, F = X_data[train_idx].shape
        
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_data[train_idx].reshape(-1, F)).reshape(N_tr, T, F)
        X_te = scaler.transform(X_data[test_idx].reshape(-1, F)).reshape(len(test_idx), T, F)
        y_tr_oh = to_categorical(y[train_idx], num_classes=N_CLASSES)
        
        model = model_builder_fn((T, F))
        model.compile(optimizer=keras.optimizers.Adam(1e-3),
                      loss='categorical_crossentropy', metrics=['accuracy'])
        
        start = time.time()
        hist = model.fit(X_tr, y_tr_oh, epochs=n_epochs, batch_size=batch_size,
                         validation_split=0.1,
                         callbacks=[
                             EarlyStopping('val_accuracy', patience=patience,
                                           restore_best_weights=True, verbose=0),
                             ReduceLROnPlateau('val_loss', factor=0.5, patience=PATIENCE,
                                               min_lr=1e-6, verbose=0)
                         ], verbose=0)
        elapsed = time.time() - start
        
        metrics = evaluate_fold(model, X_te, y[test_idx])
        metrics.update({'fold': fold_idx, 'test_subject': test_subj,
                        'history': hist.history, 'epochs': len(hist.history['loss']),
                        'time': elapsed})
        results.append(metrics)
        
        print(f"  Fold {fold_idx:2d} | S{test_subj:02d} | Acc={metrics['accuracy']*100:.2f}% | "
              f"Épocas={metrics['epochs']} | {elapsed:.0f}s")
    
    accs = [r['accuracy'] for r in results]
    print(f'  → Media: {np.mean(accs)*100:.2f}% ± {np.std(accs)*100:.2f}%')
    return results


print('Funciones listas.')

## 4. Entrenamiento GRU — Todos los joints

In [ ]:
# GRU 4 capas, todos los joints — Target: replicar 90.13% del paper
results_gru_all = train_loso_cv(
    X_data=X, y=y, S=S,
    model_builder_fn=lambda s: build_gru_model(s),
    model_name='GRU-4L (todos joints)'
)

## 5. Entrenamiento GRU — Solo piernas

In [ ]:
# GRU 4 capas, solo piernas — Target: replicar 93.67% del paper (mejor resultado)
results_gru_legs = train_loso_cv(
    X_data=X_legs, y=y, S=S,
    model_builder_fn=lambda s: build_gru_model(s),
    model_name='GRU-4L (solo piernas)'
)

## 6. Entrenamiento LSTM — Todos los joints

In [ ]:
# LSTM 4 capas, todos los joints — Target: replicar 87.25% del paper
results_lstm_all = train_loso_cv(
    X_data=X, y=y, S=S,
    model_builder_fn=lambda s: build_lstm_model(s),
    model_name='LSTM-4L (todos joints)'
)

## 7. Entrenamiento LSTM — Solo piernas

In [ ]:
results_lstm_legs = train_loso_cv(
    X_data=X_legs, y=y, S=S,
    model_builder_fn=lambda s: build_lstm_model(s),
    model_name='LSTM-4L (solo piernas)'
)

## 8. Comparación LSTM vs GRU

In [ ]:
def plot_comparison(results_dict, title, reference_values=None):
    """Visualiza comparación de modelos con accuracy por fold y promedio."""
    n_models = len(results_dict)
    n_folds = len(list(results_dict.values())[0])
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Accuracy por fold
    x = np.arange(n_folds)
    width = 0.8 / n_models
    colors = plt.cm.Set2(np.linspace(0, 1, n_models))
    
    for i, (name, results) in enumerate(results_dict.items()):
        accs = [r['accuracy']*100 for r in results]
        offset = (i - n_models/2 + 0.5) * width
        axes[0].bar(x + offset, accs, width, label=f'{name} ({np.mean(accs):.1f}%)',
                    color=colors[i], alpha=0.85, edgecolor='black', linewidth=0.5)
    
    axes[0].set_title(f'{title}\nAccuracy por Fold', fontweight='bold')
    axes[0].set_xlabel('Fold')
    axes[0].set_ylabel('Accuracy (%)')
    axes[0].legend(fontsize=9)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([f'F{i}' for i in range(n_folds)])
    axes[0].set_ylim(40, 110)
    
    # Comparación de promedios
    names = list(results_dict.keys())
    means = [np.mean([r['accuracy']*100 for r in v]) for v in results_dict.values()]
    stds  = [np.std([r['accuracy']*100 for r in v]) for v in results_dict.values()]
    
    bars = axes[1].bar(range(n_models), means, yerr=stds, color=colors,
                       edgecolor='black', capsize=5)
    
    if reference_values:
        for j, (ref_name, ref_val) in enumerate(reference_values.items()):
            axes[1].axhline(ref_val, linestyle='--', alpha=0.7, label=f'Paper: {ref_name}={ref_val}%')
        axes[1].legend(fontsize=8)
    
    axes[1].set_title('Accuracy promedio LOSO-CV\n(± desv. estándar)', fontweight='bold')
    axes[1].set_xticks(range(n_models))
    axes[1].set_xticklabels(names, rotation=15, ha='right', fontsize=9)
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_ylim(40, 110)
    for bar, mean, std in zip(bars, means, stds):
        axes[1].text(bar.get_x() + bar.get_width()/2, mean + std + 1,
                     f'{mean:.1f}%', ha='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(f'{title.replace(" ","_")}.png', dpi=100, bbox_inches='tight')
    plt.show()


# Comparación con todos los joints
plot_comparison(
    {'GRU-4L': results_gru_all, 'LSTM-4L': results_lstm_all},
    title='GRU vs LSTM — Todos los Joints',
    reference_values={'GRU paper': 90.13, 'LSTM paper': 87.25}
)

In [ ]:
# Comparación con piernas
plot_comparison(
    {'GRU-4L (piernas)': results_gru_legs, 'LSTM-4L (piernas)': results_lstm_legs},
    title='GRU vs LSTM — Solo Piernas',
    reference_values={'GRU paper (piernas)': 93.67}
)

In [ ]:
# Matrices de confusión acumuladas LOSO-CV
def plot_conf_matrix(results, title):
    cm_total = sum(r['confusion_matrix'] for r in results)
    cm_pct = cm_total.astype(float) / cm_total.sum(axis=1, keepdims=True) * 100
    cls_labels = [CLASS_NAMES[i] for i in range(N_CLASSES)]
    
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
                xticklabels=cls_labels, yticklabels=cls_labels, ax=ax,
                cbar_kws={'label': '%'})
    ax.set_title(f'Matriz de confusión — {title}', fontweight='bold')
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(f'cm_{title.replace(" ","_")}.png', dpi=100, bbox_inches='tight')
    plt.show()

plot_conf_matrix(results_gru_all,  'GRU-4L Todos Joints')
plot_conf_matrix(results_gru_legs, 'GRU-4L Solo Piernas')

In [ ]:
# Tabla comparativa completa
all_model_results = {
    'GRU-4L (todos)':    results_gru_all,
    'GRU-4L (piernas)':  results_gru_legs,
    'LSTM-4L (todos)':   results_lstm_all,
    'LSTM-4L (piernas)': results_lstm_legs,
}

print(f'\n{"Modelo":25s} {"Accuracy":>10} {"Sens.":>8} {"Espec.":>8} {"Prec.":>8}')
print('-' * 65)

# Referencia del paper
paper_refs = {
    'GRU (Jun 2020, todos)':    (90.13, '-', '-', '-'),
    'GRU (Jun 2020, piernas)':  (93.67, '-', '-', '-'),
    'LSTM (Jun 2020, todos)':   (87.25, '-', '-', '-'),
}

for name, results in all_model_results.items():
    accs = [r['accuracy'] for r in results]
    sens = np.mean([r['sensitivity'] for r in results], axis=0).mean()
    spec = np.mean([r['specificity'] for r in results], axis=0).mean()
    prec = np.mean([r['precision']   for r in results], axis=0).mean()
    print(f'{name:25s} {np.mean(accs)*100:9.2f}% {sens*100:7.2f}% {spec*100:7.2f}% {prec*100:7.2f}%')

print('-' * 65)
print('Referencias del paper (Jun et al. 2020):')
for name, (acc, *_) in paper_refs.items():
    print(f'  {name}: {acc:.2f}%')

In [ ]:
# Guardar todos los resultados
results_to_save = {}
for name, results in all_model_results.items():
    accs = [r['accuracy'] for r in results]
    results_to_save[name] = {
        'accuracies': accs,
        'mean_acc': float(np.mean(accs)),
        'std_acc': float(np.std(accs)),
        'sensitivity': np.mean([r['sensitivity'] for r in results], axis=0).tolist(),
        'specificity': np.mean([r['specificity'] for r in results], axis=0).tolist(),
        'precision': np.mean([r['precision']   for r in results], axis=0).tolist(),
        'confusion_matrix': sum(r['confusion_matrix'] for r in results).tolist(),
    }

with open('results_lstm_gru.pkl', 'wb') as f:
    pickle.dump(results_to_save, f)

print('Resultados guardados en results_lstm_gru.pkl')
print('\n→ Siguiente: Notebook 05 - BiLSTM y BiGRU')